In [9]:
import pandas as pd
import numpy as np
from joblib import Parallel, delayed


def add_basic_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.sort_values(["store_nbr", "item_nbr", "date"]).reset_index(drop=True)

    df["unit_sales_nonneg"] = df["unit_sales"].clip(lower=0)
    df["log_target"] = np.log1p(df["unit_sales_nonneg"])

    df["dayofweek"] = df["date"].dt.dayofweek
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["is_weekend"] = (df["dayofweek"] >= 5).astype("int8")

    if "onpromotion" in df.columns:
        df["onpromotion"] = (
            df["onpromotion"]
            .astype("boolean")
            .fillna(False)
            .astype("int8")
        )

    group_cols = ["store_nbr", "item_nbr"]

    for lag in [1, 7, 14, 28]:
        df[f"lag_{lag}"] = (
            df.groupby(group_cols)["unit_sales_nonneg"]
            .shift(lag)
            .astype("float32")
        )

    for window in [7, 14, 28]:
        df[f"rolling_mean_{window}"] = (
            df.groupby(group_cols)["unit_sales_nonneg"]
            .transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
            .astype("float32")
        )

    df["rolling_std_7"] = (
        df.groupby(group_cols)["unit_sales_nonneg"]
        .transform(lambda s: s.shift(1).rolling(7, min_periods=1).std())
        .astype("float32")
    )

    return df


def _corr_one_feature(df: pd.DataFrame, feature: str, target: str):
    pair = df[[feature, target]].dropna()

    if pair.empty:
        return {
            "feature": feature,
            "target": target,
            "pearson": np.nan,
            "spearman": np.nan,
            "abs_pearson": np.nan,
            "abs_spearman": np.nan,
        }

    if pair[feature].nunique() <= 1 or pair[target].nunique() <= 1:
        return {
            "feature": feature,
            "target": target,
            "pearson": np.nan,
            "spearman": np.nan,
            "abs_pearson": np.nan,
            "abs_spearman": np.nan,
        }

    pearson = pair[feature].corr(pair[target], method="pearson")
    spearman = pair[feature].corr(pair[target], method="spearman")

    return {
        "feature": feature,
        "target": target,
        "pearson": pearson,
        "spearman": spearman,
        "abs_pearson": abs(pearson) if pd.notna(pearson) else np.nan,
        "abs_spearman": abs(spearman) if pd.notna(spearman) else np.nan,
    }


def numeric_correlations_parallel(
    df: pd.DataFrame,
    target_cols=("unit_sales_nonneg", "log_target"),
    n_jobs=32,
):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    result = []

    for target in target_cols:
        features = [c for c in numeric_cols if c != target]

        rows = Parallel(n_jobs=n_jobs, backend="loky", verbose=10)(
            delayed(_corr_one_feature)(df, feature, target)
            for feature in features
        )

        result.append(pd.DataFrame(rows))

    result = pd.concat(result, ignore_index=True)
    return result.sort_values(["target", "abs_spearman"], ascending=[True, False])


def categorical_target_stats(df: pd.DataFrame, cat_cols, target="log_target", top_n=20):
    tables = {}
    for col in cat_cols:
        stat = (
            df.groupby(col, dropna=False)[target]
            .agg(["count", "mean", "median", "std"])
            .sort_values("mean", ascending=False)
            .head(top_n)
        )
        tables[col] = stat
    return tables

In [3]:
chunks = []

for chunk in pd.read_csv(
    "train.csv",
    parse_dates=["date"],
    chunksize=500_000
):
    chunk = chunk[chunk["date"] >= "2014-08-15"]
    chunks.append(chunk)

train = pd.concat(chunks, ignore_index=True)

print(train.shape)
print(train["date"].min(), train["date"].max())

/tmp/ipykernel_3298337/1907857906.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


(96423892, 6)
2014-08-15 00:00:00 2017-08-15 00:00:00


In [5]:
transactions = pd.read_csv("transactions.csv", parse_dates=["date"])
oil = pd.read_csv("oil.csv", parse_dates=["date"])
items = pd.read_csv("items.csv")
stores = pd.read_csv("stores.csv")
holidays = pd.read_csv("holidays_events.csv", parse_dates=["date"])

print(transactions.shape, oil.shape, items.shape, stores.shape, holidays.shape)

(83488, 3) (1218, 2) (4100, 4) (54, 5) (350, 6)


In [6]:
holidays_flag = holidays[["date"]].drop_duplicates().copy()
holidays_flag["is_holiday_event"] = 1

df = train.merge(transactions, on=["date", "store_nbr"], how="left")
df = df.merge(oil, on="date", how="left")
df = df.merge(items, on="item_nbr", how="left")
df = df.merge(stores, on="store_nbr", how="left")
df = df.merge(holidays_flag, on="date", how="left")

df["is_holiday_event"] = df["is_holiday_event"].fillna(0).astype("int8")
df["dcoilwtico"] = df["dcoilwtico"].ffill().bfill()

print(df.shape)
print(df.columns.tolist())
print(df.head())

(96423892, 16)
['id', 'date', 'store_nbr', 'item_nbr', 'unit_sales', 'onpromotion', 'transactions', 'dcoilwtico', 'family', 'class', 'perishable', 'city', 'state', 'type', 'cluster', 'is_holiday_event']
         id       date  store_nbr  item_nbr  unit_sales onpromotion  \
0  29073148 2014-08-15          1    103520         5.0       False   
1  29073149 2014-08-15          1    103665         7.0       False   
2  29073150 2014-08-15          1    105574         7.0       False   
3  29073151 2014-08-15          1    105575         7.0       False   
4  29073152 2014-08-15          1    105857        10.0       False   

   transactions  dcoilwtico        family  class  perishable   city  \
0        1833.0        97.3     GROCERY I   1028           0  Quito   
1        1833.0        97.3  BREAD/BAKERY   2712           1  Quito   
2        1833.0        97.3     GROCERY I   1045           0  Quito   
3        1833.0        97.3     GROCERY I   1045           0  Quito   
4        1833.0

In [7]:
df_feat = add_basic_features(df)

print(df_feat.shape)
print(df_feat[[
    "unit_sales", "unit_sales_nonneg", "log_target",
    "lag_1", "lag_7", "rolling_mean_7", "transactions", "dcoilwtico"
]].head())

(96423892, 30)
   unit_sales  unit_sales_nonneg  log_target  lag_1  lag_7  rolling_mean_7  \
0         2.0                2.0    1.098612    NaN    NaN             NaN   
1         1.0                1.0    0.693147    2.0    NaN        2.000000   
2         2.0                2.0    1.098612    1.0    NaN        1.500000   
3         3.0                3.0    1.386294    2.0    NaN        1.666667   
4         1.0                1.0    0.693147    3.0    NaN        2.000000   

   transactions  dcoilwtico  
0        1846.0       52.25  
1        1750.0       52.62  
2        1958.0       49.64  
3        1338.0       49.64  
4        1862.0       49.22  


In [10]:
corr_table = numeric_correlations_parallel(df_feat, n_jobs=32)

[Parallel(n_jobs=32)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=32)]: Done   1 tasks      | elapsed:  1.6min
[Parallel(n_jobs=32)]: Done   3 out of  24 | elapsed:  1.6min remaining: 11.1min
[Parallel(n_jobs=32)]: Done   6 out of  24 | elapsed:  2.6min remaining:  7.7min
[Parallel(n_jobs=32)]: Done   9 out of  24 | elapsed:  2.9min remaining:  4.9min
[Parallel(n_jobs=32)]: Done  12 out of  24 | elapsed:  3.7min remaining:  3.7min
[Parallel(n_jobs=32)]: Done  15 out of  24 | elapsed:  4.1min remaining:  2.4min
[Parallel(n_jobs=32)]: Done  18 out of  24 | elapsed:  5.1min remaining:  1.7min
[Parallel(n_jobs=32)]: Done  21 out of  24 | elapsed:  5.7min remaining:   48.7s
[Parallel(n_jobs=32)]: Done  24 out of  24 | elapsed:  6.1min finished
[Parallel(n_jobs=32)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=32)]: Done   1 tasks      | elapsed:  1.5min
[Parallel(n_jobs=32)]: Done   3 out of  24 | elapsed:  1.5min remaining: 10.5min
[Pa

In [11]:
print(corr_table)

              feature             target   pearson  spearman  abs_pearson  \
35  unit_sales_nonneg         log_target  0.511935  1.000000     0.511935   
27         unit_sales         log_target  0.510076  1.000000     0.510076   
46    rolling_mean_28         log_target  0.550660  0.735372     0.550660   
45    rolling_mean_14         log_target  0.540832  0.735134     0.540832   
44     rolling_mean_7         log_target  0.523416  0.727949     0.523416   
47      rolling_std_7         log_target  0.243569  0.653837     0.243569   
40              lag_1         log_target  0.397498  0.625357     0.397498   
41              lag_7         log_target  0.394743  0.614663     0.394743   
42             lag_14         log_target  0.414407  0.598166     0.414407   
43             lag_28         log_target  0.408235  0.583833     0.408235   
29       transactions         log_target  0.287777  0.272889     0.287777   
32         perishable         log_target  0.135034  0.134793     0.135034   

In [12]:
corr_log = (
    corr_table[corr_table["target"] == "log_target"]
    .sort_values("abs_spearman", ascending=False)
)

print(corr_log)

              feature      target   pearson  spearman  abs_pearson  \
35  unit_sales_nonneg  log_target  0.511935  1.000000     0.511935   
27         unit_sales  log_target  0.510076  1.000000     0.510076   
46    rolling_mean_28  log_target  0.550660  0.735372     0.550660   
45    rolling_mean_14  log_target  0.540832  0.735134     0.540832   
44     rolling_mean_7  log_target  0.523416  0.727949     0.523416   
47      rolling_std_7  log_target  0.243569  0.653837     0.243569   
40              lag_1  log_target  0.397498  0.625357     0.397498   
41              lag_7  log_target  0.394743  0.614663     0.394743   
42             lag_14  log_target  0.414407  0.598166     0.414407   
43             lag_28  log_target  0.408235  0.583833     0.408235   
29       transactions  log_target  0.287777  0.272889     0.287777   
32         perishable  log_target  0.135034  0.134793     0.135034   
28        onpromotion  log_target  0.099786  0.090922     0.099786   
39         is_weeken

In [13]:
corr_sales = (
    corr_table[corr_table["target"] == "unit_sales_nonneg"]
    .sort_values("abs_spearman", ascending=False)
)

print(corr_sales)

             feature             target   pearson  spearman  abs_pearson  \
11        log_target  unit_sales_nonneg  0.511935  1.000000     0.511935   
3         unit_sales  unit_sales_nonneg  0.996188  1.000000     0.996188   
22   rolling_mean_28  unit_sales_nonneg  0.600126  0.735372     0.600126   
21   rolling_mean_14  unit_sales_nonneg  0.604228  0.735134     0.604228   
20    rolling_mean_7  unit_sales_nonneg  0.599557  0.727949     0.599557   
23     rolling_std_7  unit_sales_nonneg  0.297936  0.653837     0.297936   
16             lag_1  unit_sales_nonneg  0.473590  0.625357     0.473590   
17             lag_7  unit_sales_nonneg  0.446426  0.614663     0.446426   
18            lag_14  unit_sales_nonneg  0.444924  0.598166     0.444924   
19            lag_28  unit_sales_nonneg  0.459611  0.583833     0.459611   
5       transactions  unit_sales_nonneg  0.139786  0.272889     0.139786   
8         perishable  unit_sales_nonneg  0.058816  0.134793     0.058816   
4        onp

In [16]:
cat_cols = [
    "family", "class", "perishable",
    "city", "state", "type", "cluster"
]

for col in cat_cols:
    stat = (
        df_feat.groupby(col, dropna=False)["log_target"]
        .agg(["count", "mean", "median", "std"])
        .sort_values("mean", ascending=False)
        .head(10)
    )
    print(f"\n=== {col} ===")
    print(stat)


=== family ===
                   count      mean    median       std
family                                                
POULTRY          1349370  2.343614  2.270578  0.989550
PRODUCE          6644791  2.264766  2.197225  1.039650
PREPARED FOODS    542790  2.036750  1.945910  0.914274
MEATS            1666138  1.997971  1.871494  0.970298
BEVERAGES       14365508  1.920452  1.791759  0.997142
EGGS             1091530  1.872981  1.791759  0.882899
BREAD/BAKERY     3436571  1.864651  1.791759  0.848492
SEAFOOD           182181  1.794599  1.791759  0.812279
DAIRY            6975162  1.726082  1.609438  0.775810
FROZEN FOODS     1122030  1.702771  1.609438  0.876722

=== class ===
        count      mean    median       std
class                                      
2786    74518  3.213634  3.258097  0.937200
2022   191668  3.069077  2.996532  1.393066
2226    47330  3.006779  2.573032  1.129677
2906     1504  2.952646  3.044522  0.941183
2904    34788  2.871865  2.928577  0.965868
6